# MCD small-model run on Colab (free T4 is enough)
Serves **Qwen3-4B-Instruct-2507** or **Llama-3.2-3B-Instruct** with vLLM, then runs the same pipeline as on your Mac.
Runtime → Change runtime type → **T4 GPU**. Run the cells in order. Re-running a cell is safe: finished conversations are skipped and half-finished ones resume.

In [ ]:
# 1. Get the project into Colab. Choose ONE source.
SOURCE = "github"          # "github" (clone a private repo) or "drive" (unzip MCD-for-colab.zip from your Drive)
GITHUB_REPO = "YOUR-USER/YOUR-REPO"      # only used for "github"

import os
from google.colab import drive
try:
    drive.mount("/content/drive")                   # results are saved here, so they survive a disconnect
    WORKSPACE = "/content/drive/MyDrive/mcd_runs/main"
except Exception as e:                              # e.g. "credential propagation was unsuccessful"
    print("!! Google Drive did not mount:", e)
    print("!! Results will be saved on Colab's TEMPORARY disk. Download them before the session ends,")
    print("!! or fix Drive (allow the pop-up, use one Google account) and re-run this cell.")
    WORKSPACE = "/content/mcd_runs/main"
os.makedirs(WORKSPACE, exist_ok=True)
print("results folder:", WORKSPACE)

if SOURCE == "github":
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")           # Colab Secrets: a read-only token for this repo only
    if not os.path.exists("/content/MCD"):
        !git clone -q https://{token}@github.com/{GITHUB_REPO}.git /content/MCD
    else:                                          # already cloned in this session: fetch your latest push
        !git -C /content/MCD pull -q
    %cd /content/MCD
else:
    if not os.path.exists("/content/drive/MyDrive/MCD-multiturn-multilang-eval"):
        !unzip -q /content/drive/MyDrive/MCD-for-colab.zip -d /content/drive/MyDrive/
    %cd /content/drive/MyDrive/MCD-multiturn-multilang-eval
!pip -q install vllm


In [ ]:
# 2. (Llama only) accept the licence on huggingface.co/meta-llama/Llama-3.2-3B-Instruct, then store HF_TOKEN in Colab Secrets
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

In [ ]:
# 3. Start the model server in the background. T4 has no bfloat16, so use half precision.
MODEL = 'Qwen/Qwen3-4B-Instruct-2507'      # or 'meta-llama/Llama-3.2-3B-Instruct'
import subprocess, time, urllib.request
subprocess.Popen(f'vllm serve {MODEL} --dtype half --max-model-len 16384 --gpu-memory-utilization 0.90 > vllm.log 2>&1', shell=True)
for _ in range(120):
    try:
        urllib.request.urlopen('http://localhost:8000/v1/models', timeout=2); print('server ready'); break
    except Exception: time.sleep(5)
else: print(open('vllm.log').read()[-2000:])

In [ ]:
# 4. One-minute check before the long run (connection, no cut-off replies, judge is skipped here)
!python -m mcd --workspace {WORKSPACE} ingest
!python -m mcd --workspace {WORKSPACE} preflight --model $MODEL --base-url http://localhost:8000/v1

### 5. Pilot run (5 conversations). It starts in the BACKGROUND so you can check progress while it works.

In [ ]:
PILOT = "LIS-S04 LIS-S09 LIS-S16 LIS-S19 LIS-S20"
!nohup python -m mcd --workspace {WORKSPACE} run --model $MODEL --base-url http://localhost:8000/v1 --conv-ids {PILOT} --conditions EN EN_REP MIX_CLEAN --workers 8 --max-tokens 4096 > {WORKSPACE}/run_pilot.log 2>&1 &
print("started in the background; run the next cell to watch it")

### 6. Watch progress. Run this cell again every minute or two (it does not wait).

In [ ]:
!tail -n 3 {WORKSPACE}/run_pilot.log
!python -m mcd --workspace {WORKSPACE} status

### 6b. (Optional) Wait here until the run has finished

In [ ]:
!while pgrep -f "mcd --workspace {WORKSPACE} run" > /dev/null; do sleep 10; done; tail -n 4 {WORKSPACE}/run_pilot.log

### 7. Full original set (finished conversations are skipped, so the pilot is not repeated)

In [ ]:
ALL_ORIGINAL = " ".join(f"LIS-S{i:02d}" for i in range(1, 21))     # LIS-S01 ... LIS-S20
!nohup python -m mcd --workspace {WORKSPACE} run --model $MODEL --base-url http://localhost:8000/v1 --conv-ids {ALL_ORIGINAL} --conditions EN EN_REP MIX_CLEAN --workers 8 --max-tokens 4096 > {WORKSPACE}/run_full.log 2>&1 &
print("started in the background; use cell 6 (change the log name to run_full.log) to watch it")

### 8. Grade the code rules; the meaning rules are exported (blind, shuffled) for a person or Claude to judge

In [ ]:
!python -m mcd --workspace {WORKSPACE} grade --judge-model manual
!python -m mcd --workspace {WORKSPACE} status

### Bring the results back
In the Files panel open `runs/main` and download `responses.jsonl` and `judge_pending.json`. Append the new `responses.jsonl` lines to your local `runs/main/responses.jsonl` (runs are keyed by model name, so models stay separate), then grade locally.

To switch models: `!pkill -f 'vllm serve'`, change `MODEL` in step 3, and repeat steps 3-8. Use the **same settings for every run of one model**.